# Day 11: React UI Logic and Full Stack
**Conditional rendering, lists and keys, loading data safely, keeping keys off the front end, and React with FastAPI**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q fastapi uvicorn httpx

# ===== Option B: VS Code / local =====
# pip install fastapi uvicorn httpx   (in your activated venv)
# For the separate React project: npm create vite@latest smartdesk-web -- --template react
print("Ready")

In [ ]:
# Renders a small React app inside the notebook output (works in Colab and VS Code notebooks).
# It loads React from a CDN, so the rendered output needs internet. The Python cell itself runs anywhere.
import html
from IPython.display import HTML, display

REACT_CDN = """<script src="https://unpkg.com/react@18.3.1/umd/react.development.js"></script>
<script src="https://unpkg.com/react-dom@18.3.1/umd/react-dom.development.js"></script>
<script src="https://unpkg.com/@babel/standalone@7.24.7/babel.min.js"></script>"""

def show_react(app_code, height=320):
    page = f"""<!doctype html><html><head>{REACT_CDN}
<style>
  body {{ font-family: system-ui, sans-serif; margin: 16px; color: #0E3B43; }}
  .card {{ background: #E3F6F5; border-radius: 10px; padding: 12px 16px; margin: 8px 0; }}
  .closed {{ opacity: 0.5; }}
  button {{ background: #0FA3A3; color: white; border: 0; border-radius: 6px; padding: 8px 14px; cursor: pointer; margin-right: 6px; }}
  input {{ padding: 8px; border: 1px solid #9ED9D4; border-radius: 6px; margin-right: 6px; }}
</style></head><body><div id="root"></div>{app_code}</body></html>"""
    display(HTML(f'<iframe srcdoc="{html.escape(page, quote=True)}" style="width:100%;height:{height}px;border:1px solid #9ED9D4;border-radius:8px"></iframe>'))

print("show_react() is ready")

## 2. Conditional rendering and lists (Session 1)

In [ ]:
show_react("""
<script type="text/babel">
const { useState } = React;

const tickets = [
  { id: 1, title: "No signal at home", priority: "high", status: "open" },
  { id: 2, title: "Change plan", priority: "low", status: "closed" },
  { id: 3, title: "Slow 5G data", priority: "medium", status: "open" },
  { id: 4, title: "Double charge", priority: "high", status: "pending" },
];
const rank = { high: 0, medium: 1, low: 2 };

function App() {
  const [loggedIn, setLoggedIn] = useState(true);
  const open = tickets.filter(t => t.status !== "closed").length;
  const sorted = [...tickets].sort((a, b) => rank[a.priority] - rank[b.priority]);
  return (
    <div>
      <p>
        {loggedIn ? "Welcome back, Mei" : "Please log in"}{" "}
        {loggedIn && open > 0 && <b>[{open} open]</b>}
      </p>
      <button onClick={() => setLoggedIn(!loggedIn)}>{loggedIn ? "Log out" : "Log in"}</button>
      {sorted.map(t => (
        <div key={t.id} className={t.status === "closed" ? "card closed" : "card"}>
          T-{100 + t.id} {t.title} ({t.priority}) {t.status === "closed" && <em>Closed</em>}
        </div>
      ))}
    </div>
  );
}
ReactDOM.createRoot(document.getElementById("root")).render(<App />);
</script>
""", height=330)

## 3. The four states with a fake API (Session 2)
The fake API waits 1 second, fails 30% of the time and sometimes returns nothing. Click Retry a few times to see every state.

In [ ]:
show_react("""
<script type="text/babel">
const { useState, useEffect } = React;

function fakeApi() {
  return new Promise((resolve, reject) => {
    setTimeout(() => {
      const r = Math.random();
      if (r < 0.3) reject(new Error("Server unavailable"));
      else if (r < 0.5) resolve([]);
      else resolve([{ id: 1, title: "No signal at home" }, { id: 2, title: "Double charge" }]);
    }, 1000);
  });
}

function Tickets() {
  const [tickets, setTickets] = useState([]);
  const [loading, setLoading] = useState(true);
  const [error, setError] = useState(null);
  const [attempt, setAttempt] = useState(0);

  useEffect(() => {
    setLoading(true);
    setError(null);
    fakeApi()
      .then(setTickets)
      .catch(err => setError(err.message))
      .finally(() => setLoading(false));
  }, [attempt]);

  const retry = <button onClick={() => setAttempt(a => a + 1)}>Retry</button>;
  if (loading) return <p>Loading tickets...</p>;
  if (error) return <div className="card">Could not load tickets: {error} {retry}</div>;
  if (tickets.length === 0) return <div className="card">No tickets yet. {retry}</div>;
  return (
    <div>
      {tickets.map(t => <div key={t.id} className="card">{t.title}</div>)}
      {retry}
    </div>
  );
}
ReactDOM.createRoot(document.getElementById("root")).render(<Tickets />);
</script>
""", height=220)

## 4. NEW (v4): Never ship API keys in front-end code
Vite replaces every `import.meta.env.VITE_...` with its value when you run `npm run build`. The result is plain text in a JavaScript file that every user downloads. The cells below **simulate** that build step in Python so you can see it happen, then scan the output for secrets. The fake key is built at runtime and is not a real key.

In [ ]:
import os, re, pathlib

FAKE_KEY = "sk-" + "demo" * 6            # not a real key; built at runtime so no key-like text is saved in files
pathlib.Path("web/src").mkdir(parents=True, exist_ok=True)
pathlib.Path("web/.env").write_text(f"VITE_API_URL=http://127.0.0.1:8000\nVITE_OPENAI_KEY={FAKE_KEY}\n")
pathlib.Path("web/src/unsafe_assist.js").write_text("""export async function suggestReply(ticket) {
  const res = await fetch("https://llm.provider.example/v1/chat", {
    headers: { Authorization: "Bearer " + import.meta.env.VITE_OPENAI_KEY },
    method: "POST", body: JSON.stringify({ prompt: ticket.title }),
  });
  return res.json();
}
""")

def fake_vite_build(src_dir, env_file, out_dir):
    """Imitates what a bundler does with VITE_ variables: it pastes their values into the output."""
    env = dict(line.split("=", 1) for line in pathlib.Path(env_file).read_text().split() if "=" in line)
    out = pathlib.Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    for f in pathlib.Path(src_dir).glob("*.js"):
        code = re.sub(r"import\.meta\.env\.(VITE_\w+)", lambda m: repr(env.get(m.group(1), "")), f.read_text())
        (out / f.name).write_text(code)
    return sorted(p.name for p in out.iterdir())

print("Built:", fake_vite_build("web/src", "web/.env", "web/dist/assets"))
print(pathlib.Path("web/dist/assets/unsafe_assist.js").read_text())

In [ ]:
SECRET_PATTERNS = {
    "API key prefix": re.compile(r"\bsk-[A-Za-z0-9_-]{16,}"),
    "Bearer token in code": re.compile(r"Bearer\s+['\"]?[A-Za-z0-9._-]{16,}"),
    "Secret-looking env name": re.compile(r"VITE_\w*(KEY|SECRET|TOKEN)\b"),
}

def scan_for_secrets(folder):
    hits = []
    for path in pathlib.Path(folder).rglob("*"):
        if path.is_file():
            for n, line in enumerate(path.read_text(errors="ignore").splitlines(), 1):
                for name, rx in SECRET_PATTERNS.items():
                    if rx.search(line):
                        hits.append(f"{path}:{n}: {name}")
    return hits or [f"no secrets found in {folder}"]

for hit in scan_for_secrets("web/dist"):
    print(hit)

### The safe pattern: the key stays on the server
React calls **your** API. FastAPI reads the key from its own environment, calls the provider, and returns only the answer. Here the provider call is an offline stand-in, so no key is needed to run it.

In [ ]:
%%writefile assist_api.py
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="SmartDesk assist (safe pattern)")

class SuggestIn(BaseModel):
    title: str = Field(min_length=1, max_length=200)

def call_llm(prompt: str) -> str:
    key = os.getenv("OPENAI_API_KEY")           # read on the server, never sent to the browser
    if not key or os.getenv("OFFLINE") == "1":
        return f"[offline demo] Suggested reply for: {prompt}"
    from openai import OpenAI                    # real call (Day 12 covers this properly)
    client = OpenAI(api_key=key)
    r = client.chat.completions.create(model=os.getenv("LLM_MODEL", "gpt-6.1-sol"),
                                       messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

@app.post("/api/tickets/{tid}/suggest")
def suggest(tid: int, body: SuggestIn):
    try:
        return {"ticket": tid, "suggestion": call_llm(f"Draft a short, polite reply about: {body.title}")}
    except Exception:
        raise HTTPException(503, "Assistant unavailable, please try again")

In [ ]:
import importlib, assist_api
importlib.reload(assist_api)
from fastapi.testclient import TestClient

os.environ["OPENAI_API_KEY"] = FAKE_KEY        # pretend the server has a key
os.environ["OFFLINE"] = "1"                    # but stay offline in this demo
api = TestClient(assist_api.app)
r = api.post("/api/tickets/7/suggest", json={"title": "Slow 5G data"})
print(r.status_code, r.json())
print("Key in response?", FAKE_KEY in r.text)
del os.environ["OPENAI_API_KEY"]

# The safe front-end code only knows your own API's address
pathlib.Path("web/src/unsafe_assist.js").unlink()
pathlib.Path("web/.env").write_text("VITE_API_URL=http://127.0.0.1:8000\n")
pathlib.Path("web/src/assist.js").write_text("""export async function suggestReply(ticket) {
  const res = await fetch(import.meta.env.VITE_API_URL + `/api/tickets/${ticket.id}/suggest`, {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ title: ticket.title }),
  });
  if (!res.ok) throw new Error(`HTTP ${res.status}`);
  return res.json();
}
""")
import shutil; shutil.rmtree("web/dist")
print("Rebuilt:", fake_vite_build("web/src", "web/.env", "web/dist/assets"))
for hit in scan_for_secrets("web/dist"):
    print(hit)

**TODO:** run `npm run build` on a real Vite project (VS Code) and run `scan_for_secrets("dist")` on it.

## 5. SmartDesk step 10: full stack tickets (Session 3)
FastAPI serves both the API (`/api/tickets`) and the React page (`/`). Because they share one origin, no CORS is needed here; CORS is still added for the separate Vite setup.

In [ ]:
%%writefile index.html
<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <title>SmartDesk</title>
  <script src="https://unpkg.com/react@18.3.1/umd/react.development.js"></script>
  <script src="https://unpkg.com/react-dom@18.3.1/umd/react-dom.development.js"></script>
  <script src="https://unpkg.com/@babel/standalone@7.24.7/babel.min.js"></script>
  <style>
    body { font-family: system-ui, sans-serif; margin: 24px; color: #0E3B43; max-width: 680px; }
    .card { background: #E3F6F5; border-radius: 10px; padding: 10px 14px; margin: 8px 0; display: flex; gap: 8px; align-items: center; }
    .closed { opacity: 0.5; text-decoration: line-through; }
    button { background: #0FA3A3; color: #fff; border: 0; border-radius: 6px; padding: 6px 12px; cursor: pointer; }
    button.close { background: #FF8A3D; margin-left: auto; }
    input, select { padding: 8px; border: 1px solid #9ED9D4; border-radius: 6px; }
  </style>
</head>
<body>
<div id="root"></div>
<script type="text/babel">
const { useState, useEffect } = React;
const API = "";   // same origin. With Vite on port 5173, use import.meta.env.VITE_API_URL instead

async function api(method, path, body) {
  const r = await fetch(API + path, { method, headers: { "Content-Type": "application/json" },
                                      body: body ? JSON.stringify(body) : undefined });
  if (!r.ok) throw new Error(`HTTP ${r.status}`);
  return r.json();
}

function SmartDesk() {
  const [tickets, setTickets] = useState([]);
  const [loading, setLoading] = useState(true);
  const [error, setError] = useState(null);
  const [title, setTitle] = useState("");
  const [saving, setSaving] = useState(false);
  const [hideClosed, setHideClosed] = useState(false);

  useEffect(() => {
    api("GET", "/api/tickets").then(setTickets).catch(e => setError(e.message)).finally(() => setLoading(false));
  }, []);

  async function create(e) {
    e.preventDefault();
    if (!title.trim()) return;
    setSaving(true);
    try { const t = await api("POST", "/api/tickets", { title }); setTickets(prev => [...prev, t]); setTitle(""); }
    catch (err) { setError(err.message); }
    setSaving(false);
  }

  async function update(t, changes) {
    const updated = await api("PATCH", `/api/tickets/${t.id}`, changes);
    setTickets(prev => prev.map(x => x.id === t.id ? updated : x));
  }

  if (loading) return <p>Loading tickets...</p>;
  if (error) return <p>Could not load tickets: {error}</p>;
  const visible = tickets.filter(t => !hideClosed || t.status !== "closed");

  return (
    <div>
      <h1>SmartDesk</h1>
      <form onSubmit={create}>
        <input value={title} onChange={e => setTitle(e.target.value)} placeholder="New ticket title" />{" "}
        <button disabled={saving}>{saving ? "Saving..." : "Create"}</button>
      </form>
      <p><label><input type="checkbox" checked={hideClosed} onChange={e => setHideClosed(e.target.checked)} /> Hide closed</label></p>
      {visible.length === 0 && <p>No tickets to show.</p>}
      {visible.map(t => (
        <div key={t.id} className="card">
          <span className={t.status === "closed" ? "closed" : ""}>T-{t.id} {t.title}</span>
          <select value={t.priority} onChange={e => update(t, { priority: e.target.value })}>
            <option>low</option><option>medium</option><option>high</option>
          </select>
          {t.status !== "closed" && <button className="close" onClick={() => update(t, { status: "closed" })}>Close</button>}
        </div>
      ))}
    </div>
  );
}
ReactDOM.createRoot(document.getElementById("root")).render(<SmartDesk />);
</script>
</body>
</html>

In [ ]:
%%writefile fullstack.py
from pathlib import Path
from typing import Literal
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

app = FastAPI(title="SmartDesk API")
app.add_middleware(CORSMiddleware, allow_origins=["http://localhost:5173", "http://127.0.0.1:5173"],
                   allow_methods=["*"], allow_headers=["*"])

Priority = Literal["low", "medium", "high"]
Status = Literal["open", "pending", "closed"]

class TicketIn(BaseModel):
    title: str = Field(min_length=1, max_length=120)
    priority: Priority = "medium"

class TicketPatch(BaseModel):
    title: str | None = Field(default=None, min_length=1, max_length=120)
    priority: Priority | None = None
    status: Status | None = None

class Ticket(BaseModel):
    id: int
    title: str
    priority: Priority = "medium"
    status: Status = "open"

tickets: dict[int, Ticket] = {
    1: Ticket(id=1, title="No signal at home", priority="high"),
    2: Ticket(id=2, title="Double charge on November bill", priority="high"),
    3: Ticket(id=3, title="Change to family plan", priority="low", status="closed"),
}

@app.get("/", response_class=HTMLResponse)
def page():
    return Path("index.html").read_text()

@app.get("/api/tickets", response_model=list[Ticket])
def list_tickets():
    return list(tickets.values())

@app.post("/api/tickets", response_model=Ticket, status_code=201)
def create(data: TicketIn):
    new_id = max(tickets, default=0) + 1
    tickets[new_id] = Ticket(id=new_id, **data.model_dump())
    return tickets[new_id]

@app.patch("/api/tickets/{tid}", response_model=Ticket)
def patch(tid: int, data: TicketPatch):
    if tid not in tickets:
        raise HTTPException(404, "Ticket not found")
    tickets[tid] = tickets[tid].model_copy(update=data.model_dump(exclude_unset=True))
    return tickets[tid]

### Test the API first

In [ ]:
import importlib, fullstack
importlib.reload(fullstack)
from fastapi.testclient import TestClient
c = TestClient(fullstack.app)

print([t["title"] for t in c.get("/api/tickets").json()])
print("Create:", c.post("/api/tickets", json={"title": "SIM not activated", "priority": "high"}).status_code)
print("Close:", c.patch("/api/tickets/1", json={"status": "closed"}).json())
print("Bad status:", c.patch("/api/tickets/1", json={"status": "done"}).status_code)
print("Missing:", c.patch("/api/tickets/99", json={"status": "closed"}).status_code)
print("Empty title:", c.post("/api/tickets", json={"title": ""}).status_code)
print("Page served:", "text/babel" in c.get("/").text)
r = c.options("/api/tickets", headers={"Origin": "http://localhost:5173", "Access-Control-Request-Method": "PATCH"})
print("CORS allowed origin:", r.headers.get("access-control-allow-origin"))

### Run the full stack app
**Colab:** the cell below starts the server and opens the app inside the notebook.

**VS Code:** run `uvicorn fullstack:app --reload` in the terminal and open http://127.0.0.1:8000 in your browser.

In [ ]:
# ===== Option A: Google Colab (active) =====
import subprocess, time
server = subprocess.Popen(["uvicorn", "fullstack:app", "--port", "8000"])
time.sleep(3)
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=500)

# ===== Option B: VS Code =====
# Terminal: uvicorn fullstack:app --reload
# Browser:  http://127.0.0.1:8000

### TODO 1: Add a `DELETE /api/tickets?status=closed` endpoint that removes closed tickets and returns the count, and a "Clear closed" button in the page.

In [ ]:
# TODO: write your code here

## 6. The separate Vite version (VS Code)
In a real project, React lives in its own Vite app. Copy the `SmartDesk` component into `src/App.jsx`, add `import { useState, useEffect } from "react";`, set `const API = import.meta.env.VITE_API_URL;` (only the API address, never a key), and export it with `export default SmartDesk;`. Run `npm run dev` and `uvicorn fullstack:app --reload` side by side. CORS is already configured for port 5173.

## 7. Check yourself
1. Which patterns did the scanner flag in the unsafe build, and why was the rebuilt bundle clean?
2. Why does the page update from the PATCH reply instead of changing state first?
3. What status code did the API return for `{"status": "done"}`, and which line caused it?

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)